# Fraud detection – CatBoost (5-seed average) + logistic regression, made robust to the test shift
Two models, each scored on its own, then rank-blended:

- **CatBoost**, averaged over 5 seeds: 24 features, native categoricals, small trees (depth 4), strong L2.
- **Logistic regression**: the lean spec (L2, C=0.3, unweighted, 3 one-hot categoricals). Velocity enters as smooth log counts (`log_tx_1h`, `log_tx_24h`), not the `burst`/`high24` flags, and it is trained with missing-value augmentation.

`prediction` is a fraud score in [0,1]: its ranking gives PR-AUC, and scores ≥ 0.5 are the fraud flags. The fitted models are saved to `model_v2.pkl`.

### Why the blend scored below CatBoost alone on the public test set
CV was ~0.24 for every model, but the public test gave ~0.19 (CatBoost 0.1925, CatBoost + logistic 50/50 0.1832). The cause is how the test data differs from training:

- **More flagged rows that look ordinary.** New-device rows go from 9% to 15%, but the extra ones look like regular customers: fewer at night (22% → 13%) and fewer at risky merchants (17% → 11%), with older accounts (median 811 → 1,544 days). Rows with 5–9 transactions in the last hour are 3–4× more common. Those bursts mostly look like legitimate high-frequency users: small amounts (median 72 vs 507 for training burst frauds), old accounts (1,038 vs 94 days), and gaming, food or transport merchants.
- **Our top ranks are built from exactly those signals.** About 80% of the top 2% are new-device rows and 60–70% are bursts, so test's extra ordinary-looking flagged rows push real frauds down.
- **Logistic regression is hit hardest.** It gives every flagged row the same fixed bonus, whatever the rest of the row looks like. Trees mostly reward flags in combination with other fraud signs. On test, 72% of logistic's top 2% are bursts (CatBoost 62%, real training frauds 21%), so blending it in pulls the ranking toward the misfiring signals.
- **Not the cause:** the big AU/JP/MY amounts (test median ×10). Those rows have old accounts, known devices and daytime card-present use, and every model already ranks them low (2–3% of the top ranks).
- **Not leakage.** No duplicate rows within train or across train/test. Imputation, scaling, encoding, CatBoost's category statistics and the augmentation are fitted on training folds only. The only things taken outside a fold are category names (training) and the shift rates for the test-like check (training and test *features*); neither uses labels.

### What this notebook changes
Each change was measured on this notebook's 5-fold CV (split seed 0, CatBoost with 1 seed for speed) on two scores: normal out-of-fold PR-AUC (clean), and PR-AUC after the test-like perturbation below (test-like). A change was kept only if it didn't lose on either.

| Change | Clean | Test-like |
|---|---|---|
| Logistic: log counts instead of the `burst`/`high24` flags (kept) | 0.2278 → 0.2303 | – |
| CatBoost **without** missing-value augmentation | 0.2322 → 0.2404 | 0.2128 → 0.2191 |
| Logistic **with** missing-value augmentation (kept) | 0.2300 → 0.2303 | 0.2067 → 0.2082 |
| Velocity counts capped at the training P99 (tried, **removed**) | CatBoost 0.2322 → 0.2259 | 0.2128 → 0.2031 |
| Logistic new-device context terms (× risky merchant, × card-present, × SG) (rejected) | 0.2221 → 0.2151 | 0.1999 → 0.1945 |
| CatBoost without the flag features (rejected) | 0.2322 → 0.2232 | 0.2128 → 0.2037 |
| CatBoost depth 3 (rejected) | 0.2404 → 0.2348 | 0.2191 → 0.2158 |
| **Final blend: CatBoost + logistic, 70/30** | **0.2442** | **0.2233** |

1. **Test-like validation.** Each validation fold is also scored after a label-blind perturbation that reproduces the measured shift merchant by merchant: extra new devices at everyday merchants, extra 5–9-per-hour bursts (gaming, food delivery, transport…), large AU/JP/MY amounts on known devices, and 3× more missing values. Only validation rows are perturbed, so no training data is affected. It drops PR-AUC by ~0.02–0.03, close to the real CV → public-test drop.
2. **Blend weight chosen on the test-like score.** Candidates are CatBoost-only and 90/10, 80/20, 70/30 and 50/50 CatBoost/logistic. The weight with the best test-like PR-AUC is used, with ties going to CatBoost.

The test-like perturbation is our model of the shift, not the real test labels, so treat that score as a robustness check, not a forecast.

In [ ]:
%pip install -q catboost scikit-learn pandas numpy

import os, sys
import numpy as np
import pandas as pd

# Read from the repo's data/ folder if it exists; otherwise open an upload button (Colab)
def load_csv(name):
    for path in [f"../data/{name}", f"data/{name}", name]:
        if os.path.exists(path):
            return pd.read_csv(path)
    from google.colab import files
    print(f"Upload: {name}")
    return pd.read_csv(list(files.upload().keys())[0])

df = load_csv("Track 2 Training Dataset.csv")
test_df = load_csv("Track 2 Testing Dataset.csv")
print("Train:", df.shape, " Test:", test_df.shape)

In [ ]:
target_column = "fraud"
CATS = ["merchant_category", "country", "transaction_channel"]

# Edge-case hardening for new data (no effect on the current files, which have no such values).
RAW_COLS = [c for c in df.columns if c not in ("id", target_column)]
NUMERIC_RAW = [c for c in RAW_COLS if c not in CATS]
NON_NEGATIVE = ["transaction_amount", "spend_last_24h", "transactions_last_1h", "transactions_last_24h", "account_age"]
# Training spelling of every category level, keyed by its trimmed lower-case form (e.g. " sg" -> "SG")
CAT_LEVELS = {c: {str(v).strip().lower(): v for v in df[c].dropna().unique()} for c in CATS}

def sanitise(d):
    d = d.reindex(columns=RAW_COLS).copy()                                 # select columns by name; missing ones become NaN
    for c in NUMERIC_RAW:
        d[c] = pd.to_numeric(d[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
    for c in NON_NEGATIVE:
        d[c] = d[c].where(d[c] >= 0)                                       # negative -> NaN
    d["transaction_hour"] = d.transaction_hour.where(d.transaction_hour.between(0, 23))
    d["new_device"] = d.new_device.where(d.new_device.isin([0, 1]))
    over = d.transactions_last_1h > d.transactions_last_24h                # 1h count can't exceed the 24h count
    d.loc[over, "transactions_last_1h"] = d.loc[over, "transactions_last_24h"]
    for c in CATS:                                                         # normalise spelling; unseen levels -> NaN -> "Unknown"
        d[c] = d[c].astype("string").str.strip().str.lower().map(CAT_LEVELS[c]).astype(object)
    return d

# Adds per-row features (uses only that row's own values, so no label or cross-row leakage)
def engineer(d):
    d = sanitise(d)
    d["log_amt"] = np.log1p(d.transaction_amount)
    d["log_spend"] = np.log1p(d.spend_last_24h)
    d["log_age"] = np.log1p(d.account_age)
    d["hour_sin"] = np.sin(2 * np.pi * d.transaction_hour / 24)
    d["hour_cos"] = np.cos(2 * np.pi * d.transaction_hour / 24)
    d["night"] = d.transaction_hour.isin([23, 0, 1, 2, 3, 4]).astype(int)
    d["burst"] = (d.transactions_last_1h >= 4).astype(float)
    d["high24"] = (d.transactions_last_24h >= 11).astype(float)
    d["young"] = (d.account_age <= 30).astype(float)
    d["amt_to_avg_ticket"] = d.log_amt - np.log1p(d.spend_last_24h / d.transactions_last_24h.clip(lower=1))
    d["share_1h"] = d.transactions_last_1h / d.transactions_last_24h.clip(lower=1)
    d["risky_merchant"] = d.merchant_category.isin(["luxury", "cash_transfer", "electronics"]).astype(int)
    d["night_x_device"] = d.night * d.new_device
    d["log_tx_1h"] = np.log1p(d.transactions_last_1h)     # smooth velocity counts for the logistic model
    d["log_tx_24h"] = np.log1p(d.transactions_last_24h)
    d["n_flags"] = d[["burst", "high24", "young", "new_device", "night", "risky_merchant"]].fillna(0).sum(axis=1)
    for c in CATS:
        d[c] = d[c].fillna("Unknown").astype("category")
    return d

# --- DROP UNNECESSARY ID COLUMNS --- (id is a random label; checked: it carries no fraud signal)
raw = df.drop(columns=["id", target_column])   # raw training columns (used by the missing-value augmentation)
X = engineer(raw)
y = df[target_column].values
NUM_COLS = [c for c in X.columns if c not in CATS]

# Test set: same features, and exactly the same category lists as training
X_test = engineer(test_df.drop(columns=["id"]))
for c in CATS:
    X_test[c] = pd.Categorical(X_test[c].astype(str), categories=X[c].cat.categories)
X_test = X_test[X.columns]
assert all("Unknown" in X[c].cat.categories for c in CATS)   # missing and unseen levels have a training category to map to
print(f"{X.shape[1]} features, fraud rate {y.mean():.2%}")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from catboost import CatBoostClassifier

# Logistic uses a lean, de-duplicated feature subset (no near-copies, no missing flags).
# Velocity enters as smooth log counts, not the burst/high24 flags, which misfire on test (fixed bonus per flagged row).
LOGIT_COLS = ["log_tx_1h", "log_tx_24h", "new_device", "log_amt", "log_age", "night", "night_x_device"]

def make_logit():
    pre = ColumnTransformer([
        # MinMaxScaler(clip=True) clips values outside the training min/max (then standardised as before)
        ("num", make_pipeline(SimpleImputer(strategy="median"), MinMaxScaler(clip=True), StandardScaler()), LOGIT_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATS),
    ])
    return make_pipeline(pre, LogisticRegression(C=0.3, max_iter=3000))

# CatBoost: small trees, strong L2, ordered target statistics for the categoricals.
# Averaged over CAT_SEEDS seeds: same model, less variance from the random seed.
CAT_SEEDS = [0, 1, 2, 3, 4]

def make_catboost(seed):
    return CatBoostClassifier(iterations=800, depth=4, learning_rate=0.03, l2_leaf_reg=10,
                              cat_features=CATS, random_seed=seed, verbose=0, allow_writing_files=False)

# CatBoost uses the original 24 features (not the logistic-only log counts), with categoricals as strings
CAT_COLS = [c for c in X.columns if c not in ("log_tx_1h", "log_tx_24h")]

def for_catboost(X_):
    return X_[CAT_COLS].astype({c: str for c in CATS})

# Probability of the fraud class
def p_fraud(model, X_):
    return model.predict_proba(X_)[:, list(model.classes_).index(1)]

# Missing-value augmentation (logistic regression only): a copy of the training rows with each value of the 8 columns that
# can be missing in test blanked with probability MASK_RATE (about the test rate). Only copies with at least one blank are
# added (~18% more rows). Labels are kept, so the model learns that a missing value is not a sign of a safe transaction.
# CatBoost is trained without it: in CV it lowered CatBoost on both clean and test-like PR-AUC (0.2404 -> 0.2322, 0.2191 -> 0.2128).
MASK_COLS = ["merchant_category", "country", "transaction_channel", "transactions_last_24h",
             "spend_last_24h", "account_age", "new_device", "transactions_last_1h"]
MASK_RATE = 0.025
# Amount and hour are never missing in the current files, but may be in new data: blanked at a lower rate.
# (Drawn after the main mask, so the blanks in the 8 columns above are the same as before.)
EXTRA_MASK_COLS = ["transaction_amount", "transaction_hour"]
EXTRA_MASK_RATE = 0.01

def add_missing_copies(raw_, y_, seed=0):
    rng = np.random.default_rng(seed)
    blank = rng.random((len(raw_), len(MASK_COLS))) < MASK_RATE
    blank_extra = rng.random((len(raw_), len(EXTRA_MASK_COLS))) < EXTRA_MASK_RATE
    copy = raw_.copy()
    copy[MASK_COLS] = copy[MASK_COLS].mask(blank)
    copy[EXTRA_MASK_COLS] = copy[EXTRA_MASK_COLS].astype(float).mask(blank_extra)
    keep = blank.any(axis=1) | blank_extra.any(axis=1)
    X_aug = engineer(pd.concat([raw_, copy[keep]], ignore_index=True))
    for c in CATS:
        X_aug[c] = pd.Categorical(X_aug[c].astype(str), categories=X[c].cat.categories)
    return X_aug[X.columns], np.concatenate([y_, y_[keep]])

# Engineered features for raw rows, with exactly the training category lists
def features(raw_):
    X_ = engineer(raw_)
    for c in CATS:
        X_[c] = pd.Categorical(X_[c].astype(str), categories=X[c].cat.categories)
    return X_[X.columns]

# Fit both models on raw training rows (raw_, y_): logistic on the augmented rows, CatBoost on the rows as they are.
def fit_models(raw_, y_):
    X_aug, y_aug = add_missing_copies(raw_, y_)
    X_ = features(raw_)
    return {"logistic": make_logit().fit(X_aug, y_aug),
             "catboost": [make_catboost(s).fit(for_catboost(X_), y_) for s in CAT_SEEDS]}

# Score with the fitted models: logistic probability, and CatBoost probability averaged over seeds
def score_models(models, X_):
    return {"logistic": p_fraud(models["logistic"], X_),
             "catboost": np.mean([p_fraud(m, for_catboost(X_)) for m in models["catboost"]], axis=0)}

# Rank-weighted blend: ranks make the models' different probability scales comparable.
# WEIGHTS is chosen in the CV cell by test-like PR-AUC; this is only the starting value.
WEIGHTS = {"catboost": 0.8, "logistic": 0.2}

def blend(scores, weights=None):
    weights = weights or WEIGHTS
    return sum(w * pd.Series(scores[name]).rank(pct=True).values for name, w in weights.items() if w > 0)

In [ ]:
from sklearn.metrics import average_precision_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold

# Test-like perturbation of VALIDATION rows only. Label-blind: rows are picked at random, never by label.
# It reproduces the measured train -> test shift, merchant by merchant, using only the test FEATURES (no test labels exist):
#  - extra new devices, mostly at everyday merchants (e.g. utilities 11% -> 35%, retail 11% -> 23%)
#  - extra rows with 5-9 transactions in the last hour (e.g. gaming 3% -> 11%, food delivery 2% -> 7%)
#  - large amounts on known devices in AU/JP/MY (test median amount x10 there)
#  - ~3x more missing values
te_raw = test_df.drop(columns=["id"])
nd_tr, nd_te = raw.groupby("merchant_category").new_device.mean(), te_raw.groupby("merchant_category").new_device.mean()
P_NEW_DEVICE = ((nd_te - nd_tr) / (1 - nd_tr)).clip(lower=0).to_dict()
fast_tr = raw.groupby("merchant_category").transactions_last_1h.apply(lambda s: (s >= 5).mean())
fast_te = te_raw.groupby("merchant_category").transactions_last_1h.apply(lambda s: (s >= 5).mean())
P_FAST = ((fast_te - fast_tr) / (1 - fast_tr)).clip(lower=0).to_dict()

def make_test_like(raw_, seed):
    rng = np.random.default_rng(seed)
    d = raw_.copy().reset_index(drop=True)
    n = len(d)
    flip = (d.new_device == 0).values & (rng.random(n) < d.merchant_category.map(P_NEW_DEVICE).fillna(0.07).values)
    d.loc[flip, "new_device"] = 1
    fast = rng.random(n) < d.merchant_category.map(P_FAST).fillna(0.024).values
    d.loc[fast, "transactions_last_1h"] = rng.integers(5, 10, fast.sum())
    d.loc[fast, "transactions_last_24h"] = np.maximum(d.loc[fast, "transactions_last_24h"].fillna(0),
                                                      d.loc[fast, "transactions_last_1h"] + rng.integers(8, 16, fast.sum()))
    big = d.country.isin(["AU", "JP", "MY"]).values & (d.new_device == 0).values & (rng.random(n) < 0.3)
    d.loc[big, "transaction_amount"] = np.minimum(d.loc[big, "transaction_amount"] * rng.uniform(5, 12, big.sum()), 9000)
    d[MASK_COLS] = d[MASK_COLS].mask(rng.random((n, len(MASK_COLS))) < 0.016)   # missing ~0.6% -> ~2.2% per column
    return features(d)

# 5-fold CV: every row is scored once by models that never saw it (out-of-fold),
# both as it is (clean) and after the test-like perturbation (shift)
oof = {"logistic": np.zeros(len(y)), "catboost": np.zeros(len(y))}
oof_shift = {"logistic": np.zeros(len(y)), "catboost": np.zeros(len(y))}
for fold, (tr, te) in enumerate(StratifiedKFold(n_splits=5, shuffle=True, random_state=0).split(X, y)):
    models_ = fit_models(raw.iloc[tr], y[tr])
    for name, s in score_models(models_, X.iloc[te]).items():
        oof[name][te] = s
    for name, s in score_models(models_, make_test_like(raw.iloc[te], seed=fold)).items():
        oof_shift[name][te] = s
    print(f"fold {fold} done")

# Choose the blend weight by test-like PR-AUC (ties go to the larger CatBoost weight)
rows = []
for w in [1.0, 0.9, 0.8, 0.7, 0.5]:
    wts = {"catboost": w, "logistic": round(1 - w, 2)}
    rows.append({"CatBoost weight": w, "clean PR-AUC": average_precision_score(y, blend(oof, wts)),
                 "test-like PR-AUC": average_precision_score(y, blend(oof_shift, wts))})
weights_table = pd.DataFrame(rows).set_index("CatBoost weight")
best_w = float(weights_table["test-like PR-AUC"].round(4).idxmax())
WEIGHTS = {"catboost": best_w, "logistic": round(1 - best_w, 2)}
print("Blend weight by test-like PR-AUC:", WEIGHTS)
print(weights_table.round(4).to_string())
oof["blend"] = blend(oof)
oof_shift["blend"] = blend(oof_shift)

# Threshold: PR-AUC does not depend on it (it only uses the ranking). It sets the 0/1 fraud flags scored by F1 / Recall.
# For every cut-off (share of rows flagged) build the confusion matrix; pick the share with the best (smoothed) F1.
def pick_share(score):
    rows = []
    for share in np.arange(0.002, 0.0601, 0.0005):
        flag = (score >= np.quantile(score, 1 - share)).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, flag).ravel()
        rows.append({"share": share, "precision": tp / max(tp + fp, 1), "recall": tp / (tp + fn), "F1": 2 * tp / (2 * tp + fp + fn)})
    cm = pd.DataFrame(rows)
    cm["F1_smooth"] = cm["F1"].rolling(5, center=True, min_periods=1).mean()   # don't pick a one-off noisy spike
    return cm.loc[cm["F1_smooth"].idxmax()]

results = []
for name, score in oof.items():
    best = pick_share(score)
    results.append({"model": name, "PR-AUC": average_precision_score(y, score),
                    "test-like PR-AUC": average_precision_score(y, oof_shift[name]), "flag share": best["share"],
                    "precision": best["precision"], "recall": best["recall"], "F1": best["F1"]})
share = pick_share(oof["blend"])["share"]
pd.DataFrame(results).set_index("model").round(4)

In [ ]:
import pickle

# Maps scores to [0,1] without changing their order (PR-AUC unchanged) and puts the flag threshold at 0.5
def to_submission_score(score, thr):
    score = np.asarray(score, dtype=float)
    lo, hi = score.min(), score.max()
    below = 0.49 * (score - lo) / max(thr - lo, 1e-12)
    above = 0.51 + 0.49 * (score - thr) / max(hi - thr, 1e-12)
    return np.clip(np.where(score >= thr, above, below), 0, 1)

# Train both models once on all 20,000 training rows and score the test set
models = fit_models(raw, y)
score = blend(score_models(models, X_test))
score = np.where(np.isfinite(score), score, np.nanmedian(score))           # defensive: a missing score gets the median
prob = to_submission_score(score, np.quantile(score, 1 - share))
prob = np.clip(np.where(np.isfinite(prob), prob, np.nanmedian(prob)), 0, 1)
sub = pd.DataFrame({"id": test_df["id"].values, "prediction": prob})
assert len(sub) == len(test_df) and sub["id"].notna().all() and sub["id"].is_unique
assert sub["prediction"].between(0, 1).all() and sub["prediction"].nunique() > 2
for path in ["../format/sample_submission.csv", "format/sample_submission.csv", "sample_submission.csv"]:
    if os.path.exists(path):                                               # ids must be in the sample file's order
        assert (pd.read_csv(path)["id"].values == sub["id"].values).all()
        print("ids match", path)
        break
sub.to_csv("submission_catboost_logistic_v2.csv", index=False)
print(f"submission_catboost_logistic_v2.csv: {len(sub)} rows, flagged (>= 0.5): {(prob >= 0.5).sum()}")

# Save the fitted models (plain sklearn + CatBoost objects) and the flag share
with open("model_v2.pkl", "wb") as f:
    pickle.dump({"models": models, "weights": WEIGHTS, "flag_share": share, "columns": list(X.columns),
                 "categories": {c: list(X[c].cat.categories) for c in CATS}}, f)
with open("model_v2.pkl", "rb") as f:
    assert np.allclose(blend(score_models(pickle.load(f)["models"], X_test)), score)   # reloads to the same scores

# In Colab: download the files
if "google.colab" in sys.modules:
    from google.colab import files
    files.download("submission_catboost_logistic_v2.csv")
    files.download("model_v2.pkl")

In [ ]:
# Edge-case check: predictions on deliberately broken inputs must be finite and in [0, 1]
edge = pd.DataFrame([
    # out-of-range values, unseen categories, 1h count above the 24h count
    {"transaction_amount": -50, "transaction_hour": 30, "merchant_category": "crypto_exchange", "country": "Mars",
     "transaction_channel": "telepathy", "transactions_last_24h": 3, "spend_last_24h": -1, "account_age": -10,
     "new_device": 7, "transactions_last_1h": 12},
    # huge values, messy spelling of known levels
    {"transaction_amount": 1e9, "transaction_hour": -1, "merchant_category": "  LUXURY ", "country": "sg",
     "transaction_channel": "ECOMMERCE", "transactions_last_24h": 10_000, "spend_last_24h": 1e12, "account_age": 1e6,
     "new_device": 1, "transactions_last_1h": 500},
    # every value missing
    {c: np.nan for c in RAW_COLS},
    # wrong types, infinity, non-binary flag, empty strings
    {"transaction_amount": "abc", "transaction_hour": 12.5, "merchant_category": None, "country": "",
     "transaction_channel": 3, "transactions_last_24h": np.inf, "spend_last_24h": np.nan, "account_age": 100,
     "new_device": 0.5, "transactions_last_1h": -3},
]).drop(columns=["spend_last_24h"])                                        # a missing column, added back as NaN

X_edge = features(edge)
assert X_edge[CATS].notna().all().all()                                    # unseen / missing levels became "Unknown"
edge_scores = score_models(models, X_edge)
edge_scores["blend"] = blend(edge_scores)
edge_scores["submission score"] = to_submission_score(edge_scores["blend"], np.quantile(score, 1 - share))
for name, s in edge_scores.items():
    assert np.isfinite(s).all() and ((s >= 0) & (s <= 1)).all(), name
print("Edge cases OK: all predictions finite and in [0, 1]")
pd.concat([X_edge[CATS].astype(str), pd.DataFrame(edge_scores).round(4)], axis=1)